In [4]:
import numpy as np
import pandas as pd

bureau = pd.read_csv("../data/raw/bureau.csv")

print("Shape:", bureau.shape)
print("Unique applicants:", bureau["SK_ID_CURR"].nunique())

print("\nRecords per applicant:")
print(bureau.groupby("SK_ID_CURR").size().describe())

print("\nFirst 5 rows:")
bureau.head()

Shape: (1716428, 17)
Unique applicants: 305811

Records per applicant:
count    305811.000000
mean          5.612709
std           4.430354
min           1.000000
25%           2.000000
50%           4.000000
75%           8.000000
max         116.000000
dtype: float64

First 5 rows:


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [5]:
def aggregate_bureau(bureau):
    b = bureau.copy()

    num_agg = b.groupby("SK_ID_CURR").agg({
        "DAYS_CREDIT": ["count", "mean", "min", "max"],
        "CREDIT_DAY_OVERDUE": ["mean", "max"],
        "AMT_CREDIT_SUM": ["sum", "mean", "max"],
        "AMT_CREDIT_SUM_DEBT": ["sum", "mean"],
        "AMT_CREDIT_SUM_OVERDUE": ["sum", "max"],
        "CNT_CREDIT_PROLONG": ["sum"],
    })

    num_agg.columns = [
        "BURO_" + "_".join(c).upper()
        for c in num_agg.columns
    ]

    active = (
        b[b["CREDIT_ACTIVE"] == "Active"]
        .groupby("SK_ID_CURR")
        .size()
    )

    closed = (
        b[b["CREDIT_ACTIVE"] == "Closed"]
        .groupby("SK_ID_CURR")
        .size()
    )

    num_agg["BURO_ACTIVE_COUNT"] = active
    num_agg["BURO_CLOSED_COUNT"] = closed

    num_agg[
        ["BURO_ACTIVE_COUNT", "BURO_CLOSED_COUNT"]
    ] = num_agg[
        ["BURO_ACTIVE_COUNT", "BURO_CLOSED_COUNT"]
    ].fillna(0)

    return num_agg.reset_index()


buro_agg = aggregate_bureau(bureau)

print("Original bureau rows:", len(bureau))
print("Aggregated rows:", len(buro_agg))
print("Unique applicants:", buro_agg["SK_ID_CURR"].nunique())
print("Aggregated columns:", buro_agg.shape[1])

buro_agg.head()

Original bureau rows: 1716428
Aggregated rows: 305811
Unique applicants: 305811
Aggregated columns: 17


,SK_ID_CURR,BURO_DAYS_CREDIT_COUNT,BURO_DAYS_CREDIT_MEAN,BURO_DAYS_CREDIT_MIN,BURO_DAYS_CREDIT_MAX,BURO_CREDIT_DAY_OVERDUE_MEAN,BURO_CREDIT_DAY_OVERDUE_MAX,BURO_AMT_CREDIT_SUM_SUM,BURO_AMT_CREDIT_SUM_MEAN,BURO_AMT_CREDIT_SUM_MAX,BURO_AMT_CREDIT_SUM_DEBT_SUM,BURO_AMT_CREDIT_SUM_DEBT_MEAN,BURO_AMT_CREDIT_SUM_OVERDUE_SUM,BURO_AMT_CREDIT_SUM_OVERDUE_MAX,BURO_CNT_CREDIT_PROLONG_SUM,BURO_ACTIVE_COUNT,BURO_CLOSED_COUNT
0,100001,7,-735.000000,-1572,-49,0.0,0,1453365.000,207623.571429,378000.0,596686.5,85240.928571,0.0,0.0,0,3.0,4.0
1,100002,8,-874.000000,-1437,-103,0.0,0,865055.565,108131.945625,450000.0,245781.0,49156.200000,0.0,0.0,0,2.0,6.0
2,100003,4,-1400.750000,-2586,-606,0.0,0,1017400.500,254350.125000,810000.0,0.0,0.000000,0.0,0.0,0,1.0,3.0
3,100004,2,-867.000000,-1326,-408,0.0,0,189037.800,94518.900000,94537.8,0.0,0.000000,0.0,0.0,0,0.0,2.0
4,100005,3,-190.666667,-373,-62,0.0,0,657126.000,219042.000000,568800.0,568408.5,189469.500000,0.0,0.0,0,2.0,1.0


In [7]:
import pandas as pd

app = pd.read_csv("../data/raw/application_train.csv")

print("Application shape:", app.shape)
print("Columns:", len(app.columns))

Application shape: (307511, 122)
Columns: 122


In [8]:
app_full = app.merge(
    buro_agg,
    on="SK_ID_CURR",
    how="left"
)

print("Original application shape:", app.shape)
print("After bureau join:", app_full.shape)

bureau_cols = [c for c in buro_agg.columns if c != "SK_ID_CURR"]

no_bureau = app_full["BURO_DAYS_CREDIT_COUNT"].isna().sum()

print("Applicants with no bureau history:", no_bureau)
print(
    "Percentage with no bureau history:",
    f"{no_bureau / len(app_full):.2%}"
)

Original application shape: (307511, 122)
After bureau join: (307511, 138)
Applicants with no bureau history: 44020
Percentage with no bureau history: 14.31%


In [10]:
RANDOM_STATE = 42

In [11]:
from sklearn.model_selection import train_test_split

X_full = app_full.drop(columns=["TARGET", "SK_ID_CURR"])
y_full = app_full["TARGET"]

X_dev_b, X_test_b, y_dev_b, y_test_b = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print("Development set:", X_dev_b.shape)
print("Test set:", X_test_b.shape)
print("Development default rate:", f"{y_dev_b.mean():.4f}")
print("Test default rate:", f"{y_test_b.mean():.4f}")

Development set: (246008, 136)
Test set: (61503, 136)
Development default rate: 0.0807
Test default rate: 0.0807


In [13]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42


def add_ratios(df):
    df = df.copy()

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    )

    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    )

    df["CREDIT_TERM"] = (
        df["AMT_ANNUITY"] / df["AMT_CREDIT"]
    )

    df["GOODS_CREDIT_RATIO"] = (
        df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
    )

    df["INCOME_PER_PERSON"] = (
        df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    )

    return df


def prep_for_lgbm(df):
    df = df.copy()

    # DAYS_EMPLOYED anomaly
    df["DAYS_EMPLOYED_ANOM"] = (
        df["DAYS_EMPLOYED"] == 365243
    ).astype(int)

    df["DAYS_EMPLOYED"] = (
        df["DAYS_EMPLOYED"].replace(365243, np.nan)
    )

    # Convert categorical columns
    for col in df.select_dtypes(exclude="number").columns:
        df[col] = df[col].astype("category")

    # Reduce memory usage
    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype("float32")

    return df


print("Functions ready.")

Functions ready.


In [14]:
X_dev_b = add_ratios(X_dev_b)
X_dev_b = prep_for_lgbm(X_dev_b)

print("Shape after preparation:", X_dev_b.shape)
print(
    "Memory:",
    round(X_dev_b.memory_usage(deep=True).sum() / 1e6, 2),
    "MB"
)
print(
    "Infinite values:",
    np.isinf(
        X_dev_b.select_dtypes(include="number")
    ).sum().sum()
)

Shape after preparation: (246008, 142)
Memory: 168.28 MB
Infinite values: 0


In [15]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import lightgbm as lgb

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

model = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    verbose=-1,
    n_jobs=2,
)

aucs = cross_val_score(
    model,
    X_dev_b,
    y_dev_b,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"Bureau: {aucs.mean():.4f} ± {aucs.std():.4f}")

Bureau: 0.7647 ± 0.0011


In [16]:
X_dev_b["EXTERNAL_DEBT_INCOME"] = (
    X_dev_b["BURO_AMT_CREDIT_SUM_DEBT_SUM"]
    / X_dev_b["AMT_INCOME_TOTAL"]
)

print(
    "Infinite values:",
    np.isinf(
        X_dev_b["EXTERNAL_DEBT_INCOME"]
    ).sum()
)

print(
    "NaN values:",
    X_dev_b["EXTERNAL_DEBT_INCOME"].isna().sum()
)

Infinite values: 0
NaN values: 35244


In [17]:
aucs = cross_val_score(
    model,
    X_dev_b,
    y_dev_b,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"Bureau + debt/income: {aucs.mean():.4f} ± {aucs.std():.4f}")

Bureau + debt/income: 0.7644 ± 0.0011


In [18]:
X_full = app_full.drop(columns=["TARGET", "SK_ID_CURR"])
y_full = app_full["TARGET"]

X_dev_b, X_test_b, y_dev_b, y_test_b = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full
)

X_dev_b = add_ratios(X_dev_b)
X_dev_b = prep_for_lgbm(X_dev_b)

print("Shape:", X_dev_b.shape)

Shape: (246008, 142)


In [19]:
bureau_results = pd.DataFrame([
    {
        "features": "application + ratios",
        "cv_auc": 0.7601,
        "cv_std": 0.0009
    },
    {
        "features": "application + ratios + bureau",
        "cv_auc": 0.7647,
        "cv_std": 0.0011
    },
    {
        "features": "application + ratios + bureau + external_debt_income",
        "cv_auc": 0.7644,
        "cv_std": 0.0011
    }
])

bureau_results

,features,cv_auc,cv_std
0,application + ratios,0.7601,0.0009
1,application + ratios + bureau,0.7647,0.0011
2,application + ratios + bureau + external_debt_...,0.7644,0.0011


In [20]:
bureau_results.to_csv(
    "../reports/bureau_experiments.csv",
    index=False
)

print("Saved: ../reports/bureau_experiments.csv")

Saved: ../reports/bureau_experiments.csv


In [21]:
del bureau
import gc
gc.collect()

print("Raw bureau data removed from memory.")

Raw bureau data removed from memory.


In [3]:
import numpy as np
import pandas as pd

app = pd.read_csv("../data/raw/application_train.csv")

print("Application shape:", app.shape)

Application shape: (307511, 122)


In [4]:
bureau = pd.read_csv("../data/raw/bureau.csv")

def aggregate_bureau(bureau):
    b = bureau.copy()

    num_agg = b.groupby("SK_ID_CURR").agg({
        "DAYS_CREDIT": ["count", "mean", "min", "max"],
        "CREDIT_DAY_OVERDUE": ["mean", "max"],
        "AMT_CREDIT_SUM": ["sum", "mean", "max"],
        "AMT_CREDIT_SUM_DEBT": ["sum", "mean"],
        "AMT_CREDIT_SUM_OVERDUE": ["sum", "max"],
        "CNT_CREDIT_PROLONG": ["sum"],
    })

    num_agg.columns = [
        "BURO_" + "_".join(c).upper()
        for c in num_agg.columns
    ]

    active = (
        b[b["CREDIT_ACTIVE"] == "Active"]
        .groupby("SK_ID_CURR")
        .size()
    )

    closed = (
        b[b["CREDIT_ACTIVE"] == "Closed"]
        .groupby("SK_ID_CURR")
        .size()
    )

    num_agg["BURO_ACTIVE_COUNT"] = active
    num_agg["BURO_CLOSED_COUNT"] = closed

    num_agg[
        ["BURO_ACTIVE_COUNT", "BURO_CLOSED_COUNT"]
    ] = num_agg[
        ["BURO_ACTIVE_COUNT", "BURO_CLOSED_COUNT"]
    ].fillna(0)

    return num_agg.reset_index()


buro_agg = aggregate_bureau(bureau)

print("Aggregated bureau shape:", buro_agg.shape)

Aggregated bureau shape: (305811, 17)


In [5]:
app_full = app.merge(
    buro_agg,
    on="SK_ID_CURR",
    how="left"
)

no_buro = app_full["BURO_DAYS_CREDIT_COUNT"].isna()

print(
    "Default rate, no bureau history:",
    round(app_full.loc[no_buro, "TARGET"].mean(), 4)
)

print(
    "Default rate, has bureau history:",
    round(app_full.loc[~no_buro, "TARGET"].mean(), 4)
)

Default rate, no bureau history: 0.1012
Default rate, has bureau history: 0.0773


In [ ]:
del bureau

print("Raw bureau data removed from memory.")

NameError: name 'bureau' is not defined